# Prototype 15 — Tap-to-Segment (SAM-family) for Add-Item Flow

**Goal:** Replace “drag a rectangle around each garment” with **tap-on-garment → mask returned → tight bbox derived from mask → cropped photo + original photo handed to Gemini**.

**Deliverable:** Model bake-off between MobileSAM and SAM 2 (tiny) on real wardrobe photos. Produces a quality grid, latency table, and decision cell.

Cross-refs:
- `15_tap_to_segment.md` — full planning doc.
- `../web/tap-to-segment/index.html` — interaction-feel prototype (HTML).
- `lib/images.py` — resize, crop, path_to_base64 utilities.
- `lib/client.py` — Gemini client for the optional `cropped + original` test in Cell 8.

> **Note:** All SAM inference runs locally via `transformers`. No API calls for the core bake-off. Cell 8 is the only one that calls Gemini (optional).

In [ ]:
import os, sys, json, time
from pathlib import Path
from dotenv import load_dotenv
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec
from PIL import Image, ImageDraw
from IPython.display import display, Markdown

load_dotenv(Path("..") / ".env")
sys.path.insert(0, str(Path("..")))

from lib.images import resize_image, crop_bbox, path_to_base64
from lib.client import VISION_MODEL, get_client

ROOT = Path("..")
INPUTS = ROOT / "inputs"
OUTPUTS = ROOT / "outputs" / "15_masks"
OUTPUTS.mkdir(parents=True, exist_ok=True)

print(f"VISION_MODEL:    {VISION_MODEL}")
print(f"GEMINI_API_KEY:  {'set' if os.getenv('GEMINI_API_KEY') else 'MISSING'}")
print(f"Output dir:      {OUTPUTS}")

## Cell 1 — Load test images

Loads the canonical bulk-add photos (`20260524_174630.jpg`, `20260524_174709.jpg`) and the `worn/` single-item shots. Resizes for display; keeps originals for mask extraction.

In [ ]:
CANONICAL_PHOTOS = [
    INPUTS / "20260524_174630.jpg",
    INPUTS / "20260524_174709.jpg",
]

WORN_DIR = INPUTS / "worn"
WORN_PHOTOS = sorted([
    p for p in WORN_DIR.glob("*.jpg") if p.is_file()
])

ALL_PHOTOS = CANONICAL_PHOTOS + WORN_PHOTOS

# Load originals + resized display copies
originals = {}
display_imgs = {}
for p in ALL_PHOTOS:
    if not p.exists():
        print(f"  SKIP (missing): {p.name}")
        continue
    img = Image.open(p)
    originals[p.name] = img
    display_imgs[p.name] = resize_image(img)
    print(f"  {p.name}: {img.width}x{img.height}  →  display {display_imgs[p.name].width}x{display_imgs[p.name].height}")

print(f"\nLoaded {len(originals)} photo(s).")

# Preview grid
n = len(originals)
cols = min(3, n)
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
if rows == 1 and cols == 1:
    axes = [[axes]]
elif rows == 1:
    axes = [axes]
elif cols == 1:
    axes = [[ax] for ax in axes]
for idx, name in enumerate(originals.keys()):
    r, c = idx // cols, idx % cols
    ax = axes[r][c]
    ax.imshow(display_imgs[name])
    ax.set_title(name, fontsize=9)
    ax.axis('off')
for idx in range(n, rows * cols):
    r, c = idx // cols, idx % cols
    axes[r][c].axis('off')
plt.tight_layout()
plt.savefig(OUTPUTS / "00_test_images.png", dpi=150)
plt.show()
print("Saved → outputs/15_masks/00_test_images.png")

## Cell 2 — Load SAM models

Loads MobileSAM (`dhkim2810/MobileSAM`) and SAM 2 tiny (`facebook/sam2-hiera-tiny`) via `transformers`. Both are Apache-2.0 licensed.

> **First run:** downloads model weights (~60 MB MobileSAM, ~200 MB SAM2-tiny). Subsequent runs use the cache.

Device auto-detection: MPS on Apple Silicon, CUDA if available, CPU fallback.

In [ ]:
import torch
from transformers import SamModel, SamProcessor

DEVICE = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

MODELS = {}
PROCESSORS = {}

model_specs = [
    ("MobileSAM", "dhkim2810/MobileSAM"),
    ("SAM2-tiny", "facebook/sam2-hiera-tiny"),
]

for name, hf_id in model_specs:
    print(f"\nLoading {name} ({hf_id})...")
    try:
        model = SamModel.from_pretrained(hf_id).to(DEVICE)
        processor = SamProcessor.from_pretrained(hf_id)
        MODELS[name] = model
        PROCESSORS[name] = processor
        print(f"  ✓ {name} loaded on {DEVICE}")
    except Exception as e:
        print(f"  ✗ {name} failed: {e}")

if not MODELS:
    print("\n⚠️  No SAM models loaded. Try: pip install transformers torch")
else:
    print(f"\n✓ {len(MODELS)} model(s) ready: {list(MODELS.keys())}")

## Cell 3 — Define tap points

For each test photo, define (x, y) tap points in pixel coordinates on the **original** image. Each tap simulates a user tapping on a garment.

Points are hand-picked by looking at Cell 1's preview grid. Edit these if the taps miss the intended garment.

In [ ]:
# Tap points: dict[photo_name] = list of (x, y, label) in original-pixel coords
# Edit before running — pick points by eyeballing Cell 1's preview

TAP_POINTS = {
    "20260524_174630.jpg": [
        # (x, y, "label") — labels are for the plot legend
        (900, 600, "white-shirt"),
        (1200, 700, "blue-jeans"),
        (1400, 300, "grey-hoodie"),
    ],
    "20260524_174709.jpg": [
        (800, 500, "black-jacket"),
        (1100, 600, "striped-top"),
        (600, 400, "beige-pants"),
    ],
}

# Auto-generate taps for worn/ single-item photos: center of image
for name, img in originals.items():
    if name.startswith("20260524"):
        continue  # already defined above
    w, h = img.size
    TAP_POINTS[name] = [(w // 2, h // 2, "item")]

print(f"Tap points defined for {len(TAP_POINTS)} photo(s):")
for name, taps in sorted(TAP_POINTS.items()):
    print(f"  {name}: {len(taps)} tap(s) → {[t[2] for t in taps]}")

## Cell 4 — Run segmentation (prompt-with-point)

For each (photo, tap, model) combo, runs SAM with a single 2D point prompt. Saves the resulting binary mask PNG to `outputs/15_masks/`.

**Structure:** `15_masks/<photo_stem>_<model>_<tapIdx>.png`

> **Time estimate:** ~2–5 s per inference on MPS; ~5–10 s on CPU. Total ~100 calls if running all combos. Skip models/photos you don't need by editing the loop.

In [ ]:
def point_prompt_inference(model, processor, image: Image.Image, point: tuple, device: str) -> np.ndarray:
    """
    Run SAM with a single foreground point prompt.
    Returns binary mask as uint8 numpy array (0 or 1).
    """
    px, py = point[0], point[1]

    inputs = processor(
        image,
        input_points=[[[px, py]]],
        input_labels=[[1]],  # 1 = foreground
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    # Get the highest-scoring mask for the single prompt
    masks = processor.image_processor.post_process_masks(
        outputs.pred_masks.cpu(),
        inputs["original_sizes"].cpu(),
        inputs["reshaped_input_sizes"].cpu(),
    )[0]  # first (only) batch item

    # masks shape: (3, H, W) — 3 candidate masks, pick best (index 0) for point prompt
    best_mask = (masks[0] > 0.5).numpy().astype(np.uint8)
    return best_mask


# Run inference for all combos
masks = {}  # {(photo_name, model_name, tap_idx): mask_array}
timings = {}  # {(photo_name, model_name, tap_idx): seconds}

total_combos = sum(
    len(taps) * len(MODELS)
    for name, taps in TAP_POINTS.items()
    if name in originals
)
done = 0

for photo_name, taps in sorted(TAP_POINTS.items()):
    if photo_name not in originals:
        print(f"  SKIP {photo_name}: not loaded")
        continue

    img = originals[photo_name]

    for tap_idx, (px, py, label) in enumerate(taps):
        for model_name in MODELS.keys():
            key = (photo_name, model_name, tap_idx)
            out_path = OUTPUTS / f"{Path(photo_name).stem}_{model_name}_tap{tap_idx}.png"

            if out_path.exists():
                print(f"  SKIP (cached) {photo_name} | {model_name} | tap {tap_idx} ({label})")
                mask = np.array(Image.open(out_path)) > 127
                masks[key] = mask.astype(np.uint8)
                timings[key] = 0.0
                done += 1
                continue

            print(f"  [{done+1}/{total_combos}] {photo_name} | {model_name} | tap {tap_idx} ({label})...", end=" ", flush=True)
            t0 = time.time()
            try:
                mask = point_prompt_inference(
                    MODELS[model_name],
                    PROCESSORS[model_name],
                    img,
                    (px, py),
                    DEVICE,
                )
                elapsed = time.time() - t0
                masks[key] = mask
                timings[key] = elapsed

                # Save mask PNG
                mask_img = Image.fromarray(mask * 255)
                mask_img.save(out_path)
                print(f"{elapsed:.2f}s  ✓")
            except Exception as e:
                print(f"ERROR: {e}")
                masks[key] = np.zeros((img.height, img.width), dtype=np.uint8)
                timings[key] = float("nan")

            done += 1

print(f"\n✓ {done}/{total_combos} combos complete. {len([t for t in timings.values() if t > 0])} with valid timings.")

## Cell 5 — Mask → tight bbox + derived crops

Derives tight bounding boxes from each mask (min/max of non-zero pixels with padding). Saves the resulting crops to `outputs/15_masks/` for side-by-side comparison.

**Output:** `15_masks/<photo_stem>_<model>_crop_tap<idx>.jpg`

In [ ]:
PADDING_PX = 20

def mask_to_bbox(mask: np.ndarray, padding: int = PADDING_PX) -> tuple:
    """
    Derive tight bounding box from binary mask.
    Returns (x, y, width, height) in pixel coords.
    """
    ys, xs = np.where(mask > 0)
    if len(ys) == 0:
        return (0, 0, mask.shape[1], mask.shape[0])  # fallback: whole image

    ymin = max(0, ys.min() - padding)
    ymax = min(mask.shape[0], ys.max() + padding)
    xmin = max(0, xs.min() - padding)
    xmax = min(mask.shape[1], xs.max() + padding)
    return (int(xmin), int(ymin), int(xmax - xmin), int(ymax - ymin))


bboxes = {}  # {(photo_name, model_name, tap_idx): (x, y, w, h)}
crops = {}   # {(photo_name, model_name, tap_idx): PIL.Image}

for (photo_name, model_name, tap_idx), mask in masks.items():
    img = originals[photo_name]
    key = (photo_name, model_name, tap_idx)

    bbox = mask_to_bbox(mask)
    bboxes[key] = bbox

    crop = img.crop((bbox[0], bbox[1], bbox[0] + bbox[2], bbox[1] + bbox[3]))
    crops[key] = crop

    crop_path = OUTPUTS / f"{Path(photo_name).stem}_{model_name}_crop_tap{tap_idx}.jpg"
    crop.save(crop_path, quality=90)

print(f"✓ Saved {len(crops)} crop(s) to {OUTPUTS}")

## Cell 6 — Quality comparison grid

Composite plot: rows = tap points, cols = models. Each cell shows the mask overlay (colored semi-transparent) on the original photo, plus the derived crop inset.

Eyeball for:
- **Mask leakage** — mask spills beyond the garment onto background
- **Missed edges** — straps, hems, collars cut off
- **Overlapping garments** — tapping one selects a different item

In [ ]:
COLORS = [
    (0.2, 0.8, 0.2, 0.35),   # green
    (0.2, 0.4, 1.0, 0.35),   # blue
    (1.0, 0.3, 0.3, 0.35),   # red
]

for photo_name in sorted(TAP_POINTS.keys()):
    if photo_name not in originals:
        continue
    taps = TAP_POINTS[photo_name]
    model_names = sorted(MODELS.keys())

    n_taps = len(taps)
    n_models = len(model_names)

    if n_models == 0:
        continue

    fig = plt.figure(figsize=(5 * n_models + 2, 4 * n_taps + 1))
    fig.suptitle(f"Mask quality — {photo_name}", fontsize=12, fontweight='bold', y=1.01)

    gs = gridspec.GridSpec(n_taps, n_models, figure=fig, hspace=0.3, wspace=0.3)

    for tap_idx in range(n_taps):
        tap_xy = taps[tap_idx][:2]
        tap_label = taps[tap_idx][2]

        for m_idx, model_name in enumerate(model_names):
            ax = fig.add_subplot(gs[tap_idx, m_idx])
            key = (photo_name, model_name, tap_idx)

            if key not in masks:
                ax.text(0.5, 0.5, 'N/A', ha='center', va='center', transform=ax.transAxes)
                ax.set_title(f"{model_name}\n{tap_label}" if m_idx == 0 else model_name, fontsize=8)
                continue

            # Show original with mask overlay
            img = display_imgs[photo_name]
            # Resize mask to display dimensions
            mask_display = np.array(Image.fromarray(masks[key] * 255).resize(
                (display_imgs.get(photo_name, originals[photo_name]).width,
                 display_imgs.get(photo_name, originals[photo_name]).height),
                Image.NEAREST
            )) > 127

            color = COLORS[tap_idx % len(COLORS)]

            ax.imshow(img if photo_name in display_imgs else originals[photo_name])

            # Overlay mask as colored semi-transparent layer
            overlay = np.zeros((*mask_display.shape, 4))
            overlay[mask_display] = color[:3] + (0.4,)
            ax.imshow(overlay, alpha=0.5)

            # Draw tap point
            # Scale tap point to display coords
            scale_x = (display_imgs.get(photo_name, originals[photo_name]).width) / originals[photo_name].width if photo_name in originals else 1
            scale_y = (display_imgs.get(photo_name, originals[photo_name]).height) / originals[photo_name].height if photo_name in originals else 1
            ax.plot(tap_xy[0] * scale_x, tap_xy[1] * scale_y, 'wo', markersize=4, markeredgecolor='black', markeredgewidth=0.5)

            # Draw bbox
            bbox = bboxes[key]
            bbox_scaled = (bbox[0] * scale_x, bbox[1] * scale_y, bbox[2] * scale_x, bbox[3] * scale_y)
            rect = patches.Rectangle(
                (bbox_scaled[0], bbox_scaled[1]),
                bbox_scaled[2], bbox_scaled[3],
                linewidth=1.5, edgecolor=color[:3], facecolor='none', linestyle='--'
            )
            ax.add_patch(rect)

            title = f"{model_name}"
            if m_idx == 0:
                title += f"\n{tap_label}"
            ax.set_title(title, fontsize=8)
            ax.axis('off')

    plt.tight_layout()
    plt.savefig(OUTPUTS / f"{Path(photo_name).stem}_quality_grid.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved → outputs/15_masks/{Path(photo_name).stem}_quality_grid.png")

## Cell 7 — Latency benchmarks

Per-model wall-clock timings. Prints a markdown table for easy copy into the spec. On MPS, expect ~0.3–0.8 s for MobileSAM, ~0.5–1.5 s for SAM2-tiny.

In [ ]:
from collections import defaultdict

model_timings = defaultdict(list)

for (photo_name, model_name, tap_idx), elapsed in timings.items():
    if elapsed > 0:
        model_timings[model_name].append(elapsed)

print(f"Device: {DEVICE}\n")
print("| Model | Runs | Mean | Median | Min | Max |")
print("|-------|------|------|--------|-----|-----|")

for model_name in sorted(model_timings.keys()):
    ts = model_timings[model_name]
    print(f"| {model_name} | {len(ts)} | {np.mean(ts):.3f}s | {np.median(ts):.3f}s | {np.min(ts):.3f}s | {np.max(ts):.3f}s |")

print()
for model_name in sorted(model_timings.keys()):
    ts = model_timings[model_name]
    plt.hist(ts, alpha=0.6, bins=15, label=model_name)
plt.xlabel("Seconds")
plt.ylabel("Count")
plt.title(f"SAM inference latency ({DEVICE})")
plt.legend()
plt.savefig(OUTPUTS / "latency_histogram.png", dpi=150)
plt.show()
print("Saved → outputs/15_masks/latency_histogram.png")

## Cell 8 — [OPTIONAL] cropped + original → Gemini compare

Tests whether feeding both `cropped photo` + `original photo` to Gemini improves categorization quality compared to the current single-crop pipeline. Runs once on a representative item.

> **Only run this after the bake-off cells produce good masks.** Costs 1 Gemini call.

In [ ]:
SKIP_GEMINI_TEST = True  # ← set to False to run

if SKIP_GEMINI_TEST or not os.getenv("GEMINI_API_KEY"):
    print("Skipped. Set SKIP_GEMINI_TEST = False and ensure GEMINI_API_KEY is set.")
else:
    from lib.categorize import categorize_item

    # Pick first photo + first tap as the test case
    TEST_PHOTO = sorted(TAP_POINTS.keys())[0]
    TEST_MODEL = sorted(MODELS.keys())[0]
    key = (TEST_PHOTO, TEST_MODEL, 0)

    if key in crops:
        original_b64 = path_to_base64(INPUTS / TEST_PHOTO)
        crop_b64 = path_to_base64(OUTPUTS / f"{Path(TEST_PHOTO).stem}_{TEST_MODEL}_crop_tap0.jpg")

        print(f"Testing on {TEST_PHOTO} with {TEST_MODEL}")
        print("--- single-crop result ---")
        result_single = categorize_item(crop_b64)
        print(json.dumps(result_single, indent=2))

        print("\n--- cropped + original result ---")
        # Combine: send both images — Gemini sees crop first, then original
        # (categorize_item only takes one image, so we call the model directly)
        client = get_client()
        prompt = (
            "You are a clothing categorization expert. Analyze this garment.\n"
            "The first image is a cropped close-up of a garment.\n"
            "The second image is the original photo showing the garment in context.\n"
            "Return JSON: { \"category\": string, \"color\": string, \"material\": string, "
            "\"subcategory\": string, \"wear_context\": string }"
        )
        response = client.models.generate_content(
            model=VISION_MODEL,
            contents=[
                prompt,
                {"inline_data": {"mime_type": "image/jpeg", "data": crop_b64}},
                {"inline_data": {"mime_type": "image/jpeg", "data": original_b64}},
            ],
        )
        print(response.text)

        # Side-by-side display
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
        ax1.imshow(crops[key])
        ax1.set_title("Cropped (single input)")
        ax1.axis('off')
        ax2.imshow(originals[TEST_PHOTO])
        ax2.set_title("Original (context)")
        ax2.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f"No crop found for {key}. Run Cell 4+5 first.")

## Cell 9 — Decision cell

**After reviewing the quality grid and latency table**, fill in the decision here.

**Decision criteria:**
1. Mask quality on canonical bulk-add photos (the messy-room shots).
2. Failure modes on adversarial cases (thin straps, overlapping garments, same-colour-on-same-colour).
3. Latency on the target device class (MPS for prototyping; WASM/WebGPU for real deployment).
4. Model size (download time matters for PWA).

**Default pick:** MobileSAM unless SAM2-tiny is clearly better *and* not noticeably slower.

In [ ]:
CHOSEN_MODEL = None  # ← fill in after reviewing the grid
REASONING = """
Replace this with the reasoning behind the model choice.
Include specific observations from the quality grid: leakage, strap handling, overlapping-garment disambiguation.
"""

if CHOSEN_MODEL:
    display(Markdown(f"**Chosen model:** `{CHOSEN_MODEL}`"))
    display(Markdown(REASONING))
else:
    print("⚠️  Decision pending — review the quality grid and fill in CHOSEN_MODEL + REASONING above.")

## Done criteria

Prototype is "done enough to spec a branch" when:

- [x] Quality grid saved to `outputs/15_masks/` for each photo
- [x] Latency table printed above
- [ ] Decision cell filled in with chosen model + reasoning
- [ ] (Optional) Cell 8 shows cropped+original → Gemini result

Promote relevant findings into `notes/ai-generated/segmentation-3d-video-research.md` §1 once these hold.